# Analysis Notebook

Reproduces the figures and tables in `docs/experiments.md` from the JSON that
the evaluation scripts write.

**Every cell reads measured output.** Nothing here estimates or interpolates a
value: where a run has not been performed, the cell reports `NOT MEASURED`.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

OUT = ROOT / 'outputs'

def load(path):
    """Load a JSON artefact, returning None when the run has not been performed."""
    p = ROOT / path
    if not p.exists():
        print(f'NOT MEASURED - {path} does not exist')
        return None
    return json.loads(p.read_text())

print('root:', ROOT)

## 1. Measured architecture facts

Probed from the loaded networks, not read from documentation.


In [ ]:
import subprocess

info = load('outputs/model_inspection.json')
if info is None:
    print('Run: python scripts/inspect_model.py')
else:
    for name, m in info['models'].items():
        print(f"{name:<22} {m['parameters']:>12,} params  {m['gflops']} GFLOPs  taps {m['kd_layers']}")
        print(f"{'':22} channels {m['feature_channels']}  cal_b={m['calibration']['cal_b']:.4f}")

## 2. The alignment gap

The central methodological result: median alignment hides absolute-scale error,
which is exactly the error obstacle distance depends on.


In [ ]:
d = load('outputs/evaluation/depth_metrics.json')

if d is not None and 'metric' in d:
    keys = ['abs_rel', 'rmse', 'delta1', 'delta2', 'delta3']
    metric = [d['metric'][k] for k in keys]
    aligned = [d['aligned'][k] for k in keys]

    x = np.arange(len(keys))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.2, metric, 0.4, label='MODE 1 metric (align=none)')
    ax.bar(x + 0.2, aligned, 0.4, label='MODE 2 aligned (align=median)')
    ax.set_xticks(x, keys)
    ax.set_ylabel('value')
    ax.set_title('Metric vs aligned evaluation')
    ax.legend(frameon=False)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.show()

    ratio = d['aligned']['delta1'] / max(d['metric']['delta1'], 1e-9)
    print(f"delta1 inflation from alignment: {ratio:.2f}x")
else:
    print('Run: python evaluation/evaluate_depth.py --model <ckpt> --data <data.yaml>')

## 3. Per-stage latency

P99, not the mean, governs the stale-frame timeout.


In [ ]:
lat = load('outputs/evaluation/latency_benchmark.json') or load('outputs/demo/latency.json')

if lat is not None and 'pipeline_stages' in lat:
    stages = {k: v for k, v in lat['pipeline_stages'].items() if 'mean_ms' in v}
    names = list(stages)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(names, [stages[n]['mean_ms'] for n in names], label='mean')
    ax.plot(names, [stages[n]['p99_ms'] for n in names], 'o--', color='crimson', label='P99')
    ax.set_ylabel('latency (ms)')
    ax.set_title(f"Per-stage latency on {lat.get('device', 'unknown')}")
    ax.legend(frameon=False)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.show()

    total = stages.get('total', {})
    if total:
        print(f"end-to-end mean {total['mean_ms']:.2f} ms -> {1000 / total['mean_ms']:.2f} FPS")
else:
    print('Run: python evaluation/benchmark_latency.py')

## 4. Obstacle distance error

The signed bias matters more than the magnitude: over-estimating distance is the
direction that causes collisions.


In [ ]:
dist = load('outputs/evaluation/distance_metrics.json')

if dist is not None and dist.get('num_obstacles'):
    for k in ['obstacle_distance_mae', 'obstacle_distance_rmse', 'obstacle_distance_mape',
              'percentage_within_10_percent', 'percentage_within_20_percent',
              'percentage_within_30_percent', 'mean_signed_error', 'overestimated_fraction']:
        if k in dist:
            print(f'{k:<32}{dist[k]:>10.4f}')
    print()
    print('ground truth:', dist.get('ground_truth_definition', 'unspecified'))
else:
    print('Run: python evaluation/evaluate_distance.py --model <ckpt> --data <data.yaml>')

## 5. Ablation table

Cells reading `NOT MEASURED` correspond to runs that were not performed. No value
is invented to fill them.


In [ ]:
p = ROOT / 'docs' / 'ablation.md'
if p.exists():
    print(p.read_text())
else:
    print('NOT MEASURED - run: python evaluation/ablation.py --experiments outputs/experiments')

## 6. Optuna study


In [ ]:
a = load('outputs/optuna/analysis.json')

if a is not None:
    print(f"study {a['study']}: {a['n_complete']}/{a['n_trials']} trials complete")
    print(f"best trial {a['best_trial']} scored {a['best_score']:.4f}")
    for k, v in a['best_params'].items():
        print(f'  {k:<24}{v}')

    imp = a.get('param_importances') or {}
    if imp:
        ks = sorted(imp, key=imp.get)
        fig, ax = plt.subplots(figsize=(7, max(2, 0.4 * len(ks))))
        ax.barh(ks, [imp[k] for k in ks])
        ax.set_xlabel('importance')
        ax.set_title(f"Parameter importance ({a.get('importance_method', 'unknown')})")
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)
        plt.tight_layout()
        plt.show()
else:
    print('Run: python tuning/analyze_trials.py --storage sqlite:///outputs/optuna/kd_depth_search.db')

## 7. Reproducibility record


In [ ]:
meta = load('outputs/experiment_metadata.json')
if meta is not None:
    for k, v in meta.items():
        print(f'{k:<24}{v}')
else:
    print('NOT MEASURED - written automatically by any training script')